# Edge TinyML — Recyclables Classifier (Full Analysis)
**Student:** Remmy Kipruto Tumo

This Colab-ready notebook includes: training with transfer learning (MobileNetV2), TFLite conversion
with integer quantization, evaluation with confusion matrix and per-class metrics, latency measurements
(simulated for TFLite), and a SHAP explainability snippet for model interpretability.

Instructions: Upload your dataset to `/content/data` with `train/`, `val/`, and `test/` subfolders, each containing class subfolders.


In [ ]:
# %%bash
# Install required packages (uncomment when running in Colab)
# Note: running these installs inside other environments may require adjustments.
pip_install_cmds = [
    "pip install -q tensorflow==2.12.0",
    "pip install -q scikit-learn matplotlib seaborn shap",
    "pip install -q tflite-runtime || true"
]
print('\n'.join(pip_install_cmds))


In [ ]:
# Imports and basic configuration
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import time
print('TensorFlow version:', tf.__version__)


In [ ]:
# Dataset parameters - adjust as needed
DATA_DIR = '/content/data'  # upload dataset here
IMG_SIZE = (160,160)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE


In [ ]:
# Load datasets (train / val / test)
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    os.path.join(DATA_DIR, 'train'), image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical')
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    os.path.join(DATA_DIR, 'val'), image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical')
test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    os.path.join(DATA_DIR, 'test'), image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical')

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print('Classes:', class_names)
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)


In [ ]:
# Build the model using MobileNetV2 as a backbone (transfer learning)
base_model = tf.keras.applications.MobileNetV2(input_shape=IMG_SIZE+(3,), include_top=False, weights='imagenet')
base_model.trainable = False
inputs = tf.keras.Input(shape=IMG_SIZE+(3,))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
# Train the model
EPOCHS = 10
history = model.fit(train_ds, epochs=EPOCHS, validation_data=val_ds)


In [ ]:
# Optional: small fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
history_ft = model.fit(train_ds, epochs=5, validation_data=val_ds)


In [ ]:
# Evaluate on test set and create predictions
loss, acc = model.evaluate(test_ds)
print('Test accuracy:', acc)

# gather true labels and preds for confusion matrix
y_true = []
y_pred = []
for imgs, labels in test_ds:
    preds = model.predict(imgs)
    y_true.extend(np.argmax(labels.numpy(), axis=1).tolist())
    y_pred.extend(np.argmax(preds, axis=1).tolist())

print('Collected', len(y_true), 'test samples')


In [ ]:
# Confusion Matrix and Classification Report
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

print('\nClassification Report:\n')
print(classification_report(y_true, y_pred, target_names=class_names))


In [ ]:
# Save Keras model
MODEL_H5 = 'model_recyclables.h5'
model.save(MODEL_H5)
print('Saved', MODEL_H5)


In [ ]:
# Convert to TFLite with integer quantization (representative dataset)
def representative_data_gen():
    for images, labels in train_ds.take(100):
        batch = tf.image.resize(images, IMG_SIZE)
        batch = tf.cast(batch, tf.float32)
        yield [batch.numpy()]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8
tflite_model = converter.convert()
TFLITE_PATH = 'model_recyclables_quant.tflite'
open(TFLITE_PATH, 'wb').write(tflite_model)
print('Saved', TFLITE_PATH)


In [ ]:
# Quick TFLite inference test and latency measurement (simulated in Colab)
interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

def tflite_predict(interpreter, input_array):
    input_index = input_details[0]['index']
    output_index = output_details[0]['index']
    interpreter.set_tensor(input_index, input_array)
    start = time.time()
    interpreter.invoke()
    latency = (time.time() - start) * 1000
    preds = interpreter.get_tensor(output_index)
    return preds, latency

# measure latency over 50 samples from test set
latencies = []
preds_list = []
for imgs, labels in test_ds.take(5):
    # run per-image
    for img in imgs:
        img_resized = tf.image.resize(img, IMG_SIZE)
        img_uint8 = tf.cast(img_resized, tf.uint8).numpy()[np.newaxis, ...]
        preds, lat = tflite_predict(interpreter, img_uint8)
        latencies.append(lat)
        preds_list.append(np.argmax(preds[0]))

print('TFLite simulated latency samples (ms):')
print('mean:', np.mean(latencies), 'median:', np.median(latencies), '95th:', np.percentile(latencies, 95))


In [ ]:
# SHAP explainability snippet (useful for small datasets / tabular features)
# For deep models, DeepExplainer can be used but it requires a TF model and may be slow.
import shap

# Create a small background dataset (take few training images)
bg_images = []
for imgs, labels in train_ds.take(10):
    for img in imgs:
        bg_images.append(img.numpy())
        if len(bg_images) >= 50:
            break
    if len(bg_images) >= 50:
        break
bg = np.array(bg_images)

# Use GradientExplainer or DeepExplainer depending on TF version
try:
    explainer = shap.GradientExplainer((model.layers[0].input, model.layers[-1].output), bg)
    # pick one test image
    test_img = next(iter(test_ds.take(1)))[0][0].numpy()[np.newaxis, ...]
    shap_values = explainer.shap_values(test_img)
    print('Computed SHAP values shape:', [s.shape for s in shap_values])
except Exception as e:
    print('SHAP explainability failed or is slow in this environment:', e)
    print('If running in Colab, ensure shap is installed and use DeepExplainer/GradientExplainer accordingly.')


## Next steps and notes
- After running the notebook, update `reports/task1_report.md` with the measured accuracy, model size, and latency values.
- For Raspberry Pi deployment, prefer `tflite-runtime` for smaller footprint. On Pi measure latency using `time.time()` around `interpreter.invoke()`.
- If you want extreme compression for microcontrollers, investigate pruning, knowledge distillation, and TFLite Micro.
- SHAP on image models is computationally heavy; consider using occlusion sensitivity or Grad-CAM for visual explanations as alternatives.
